
# Four-field class-conditional quickcheck

This notebook is for the proposed class-conditional runs that remove `Mstar` and `Mtot` and keep only:

`Mcdm, HI, Mgas, ne`

It does four things:

1. Check Slurm/job/log status for the submitted jobs.
2. Audit whether the actual run names are really four-field runs.
3. Check latest checkpoints and DPM50 sample files.
4. Plot one-point PDFs, auto-power ratios, and physically meaningful cross-correlation curves when matched class-only samples exist.

Important: if a job log says `mcdm-mstar-hi-mgas-mtot-ne`, then that job is the old six-field run, even if the submission command set `NF_CLASS_FIELDS='Mcdm,HI,Mgas,ne'`. In that case, pull the env-aware config-generator update before rerunning training.


In [ ]:

from __future__ import annotations

import json
import math
import os
import re
import subprocess
from pathlib import Path
from typing import Any

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import yaml

from simdiff_eval.metrics import batch_power_spectra

plt.rcParams.update({
    'font.size': 13,
    'axes.titlesize': 16,
    'axes.labelsize': 14,
    'legend.fontsize': 11,
    'xtick.labelsize': 12,
    'ytick.labelsize': 12,
})

DEFAULT_PROJECT = Path('/home/jiamingp/diffusion_models_repo')
PROJECT_DIR = Path(os.environ.get('PROJECT_DIR', DEFAULT_PROJECT)).expanduser()
if not PROJECT_DIR.exists():
    PROJECT_DIR = Path.cwd().resolve()
    if PROJECT_DIR.name == 'notebooks':
        PROJECT_DIR = PROJECT_DIR.parent

SWEEP = 'nf_class_conditional_u128'
LOG_DIR = PROJECT_DIR / 'logs' / SWEEP
RESULT_DIR = PROJECT_DIR / 'results' / SWEEP
SAMPLE_DIR = RESULT_DIR / 'samples'
OUT_DIR = PROJECT_DIR / 'results' / 'conditional_inference_fourfield'
OUT_DIR.mkdir(parents=True, exist_ok=True)

JOB_IDS = [int(x) for x in os.environ.get('NF_CLASS_JOB_IDS', '51234305,51234317').replace(' ', '').split(',') if x]
SAMPLE_LABEL = os.environ.get('CLASS_SAMPLE_LABEL', 'dpm50_class_conditional')
SEED = int(os.environ.get('CLASS_SAMPLE_SEED', '123'))
REAL_SIMS_PER_FIELD = int(os.environ.get('CLASS_REAL_SIMS_PER_FIELD', '16'))
PK_NBINS = int(os.environ.get('CLASS_PK_NBINS', '25'))
PLOT_ORDER_FALLBACK_CROSS = os.environ.get('PLOT_ORDER_FALLBACK_CROSS', '0') == '1'

EXPECTED_FIELDS = [x.strip() for x in os.environ.get('NF_CLASS_EXPECTED_FIELDS', 'Mcdm,HI,Mgas,ne').split(',') if x.strip()]
EXPECTED_FIELD_SET = set(EXPECTED_FIELDS)
REMOVED_FIELDS = {'Mstar', 'Mtot'}

PREFERRED_RUNS = [
    'nf_class_u128_mcdm-hi-mgas-ne_lh_z0p0_n1000_logfloor_perfieldnorm_400k',
    'nf_class_u128_mcdm-hi-mgas-ne_lh_z0p0_n128_logfloor_perfieldnorm_200k',
]

print('PROJECT_DIR:', PROJECT_DIR)
print('jobs:', JOB_IDS)
print('expected fields:', EXPECTED_FIELDS)
print('sample label:', SAMPLE_LABEL)
print('output:', OUT_DIR)


In [ ]:

FIELD_CANON = {
    'mcdm': 'Mcdm',
    'mstar': 'Mstar',
    'hi': 'HI',
    'mgas': 'Mgas',
    'mtot': 'Mtot',
    'ne': 'ne',
}


def run_shell(args: list[str]) -> str:
    try:
        out = subprocess.check_output(args, text=True, stderr=subprocess.STDOUT)
        return out.strip()
    except Exception as exc:
        return f'COMMAND FAILED: {exc}'


def sacct_df(job_ids: list[int]) -> pd.DataFrame:
    if not job_ids:
        return pd.DataFrame()
    raw = run_shell([
        'sacct',
        '-j', ','.join(str(j) for j in job_ids),
        '--format=JobID,JobName%24,Account,State,ExitCode,Elapsed,MaxRSS',
        '--parsable2',
    ])
    if raw.startswith('COMMAND FAILED') or not raw:
        print(raw)
        return pd.DataFrame()
    lines = raw.splitlines()
    header = lines[0].split('|')
    rows = [line.split('|') for line in lines[1:] if line.strip()]
    return pd.DataFrame(rows, columns=header)


def parse_train_log(job_id: int) -> dict[str, Any]:
    out_path = LOG_DIR / f'train_{job_id}.out'
    err_path = LOG_DIR / f'train_{job_id}.err'
    out = out_path.read_text(errors='replace') if out_path.exists() else ''
    err = err_path.read_text(errors='replace') if err_path.exists() else ''
    run_match = re.findall(r'Training discrete class-conditional\s+(\S+)', out)
    epochs = [(int(e), float(v)) for e, v in re.findall(r'Epoch\s+(\d+)\s+—\s+avg loss:\s+([0-9.eE+-]+)', out)]
    ckpts = re.findall(r'Checkpoint saved\s+→\s+(\S+)', out)
    return {
        'job_id': job_id,
        'out_exists': out_path.exists(),
        'err_exists': err_path.exists(),
        'run_name_from_log': run_match[-1] if run_match else None,
        'epoch_min': min([e for e, _ in epochs]) if epochs else None,
        'epoch_max': max([e for e, _ in epochs]) if epochs else None,
        'last_loss': epochs[-1][1] if epochs else None,
        'n_epoch_lines': len(epochs),
        'last_checkpoint': ckpts[-1] if ckpts else None,
        'training_complete': 'Training complete.' in out,
        'time_limit': 'DUE TO TIME LIMIT' in err,
        'nonfinite': 'Non-finite' in out or 'Non-finite' in err,
        'out_path': str(out_path),
        'err_path': str(err_path),
    }


def fields_from_run_name(run_name: str | None) -> list[str]:
    if not run_name:
        return []
    m = re.search(r'nf_class_u128_(.*?)_lh_', str(run_name))
    if not m:
        return []
    return [FIELD_CANON.get(token.lower(), token) for token in m.group(1).split('-')]


def audit_fields(run_name: str | None, manifest_row: dict[str, Any] | None = None) -> dict[str, Any]:
    manifest_fields = list((manifest_row or {}).get('fields') or [])
    fields = manifest_fields or fields_from_run_name(run_name)
    field_set = set(fields)
    expected_ok = field_set == EXPECTED_FIELD_SET
    removed_present = sorted(field_set & REMOVED_FIELDS)
    if expected_ok:
        verdict = 'OK: intended four-field run'
    elif removed_present:
        verdict = 'WRONG: contains removed fields; this is not the intended four-field run'
    else:
        verdict = 'CHECK: field set differs from expected four-field run'
    return {
        'fields_detected': ','.join(fields),
        'matches_expected_fourfield': expected_ok,
        'removed_fields_present': ','.join(removed_present),
        'verdict': verdict,
    }


def read_manifest() -> list[dict[str, Any]]:
    path = PROJECT_DIR / 'local' / SWEEP / 'manifest.json'
    if not path.exists():
        print('missing manifest:', path)
        return []
    loaded = json.loads(path.read_text())
    return [loaded] if isinstance(loaded, dict) else list(loaded)


def latest_checkpoint_epoch(checkpoint_dir: str | Path | None) -> tuple[int | None, str | None]:
    if not checkpoint_dir:
        return None, None
    d = Path(checkpoint_dir)
    if not d.exists():
        return None, None
    best_epoch = None
    best_path = None
    for p in d.glob('checkpoint-*'):
        m = re.search(r'(\d+)$', p.name)
        if not m:
            continue
        epoch = int(m.group(1))
        if best_epoch is None or epoch > best_epoch:
            best_epoch = epoch
            best_path = str(p)
    return best_epoch, best_path


def sample_path_for_run(run_name: str, label: str = SAMPLE_LABEL, seed: int = SEED) -> Path:
    return SAMPLE_DIR / f'{run_name}_seed{seed}_{label}.npz'


In [ ]:

print('sacct status:')
sacct = sacct_df(JOB_IDS)
display(sacct)

log_rows = [parse_train_log(j) for j in JOB_IDS]
log_df = pd.DataFrame(log_rows)
print('parsed training logs:')
display(log_df)

manifest = read_manifest()
manifest_by_run = {row.get('run_name'): row for row in manifest}

log_audit_rows = []
for row in log_rows:
    run_name = row.get('run_name_from_log')
    audit = audit_fields(run_name, manifest_by_run.get(run_name))
    log_audit_rows.append({
        'job_id': row['job_id'],
        'run_name_from_log': run_name,
        **audit,
        'training_complete': row.get('training_complete'),
        'epoch_max': row.get('epoch_max'),
        'last_loss': row.get('last_loss'),
    })
log_audit_df = pd.DataFrame(log_audit_rows)
print('field audit from logs:')
display(log_audit_df)

if len(log_audit_df) and not log_audit_df['matches_expected_fourfield'].fillna(False).all():
    print('\nSTOP: at least one submitted job is not the intended four-field run.')
    print('The pasted 51234305/51234317 logs show mstar/mtot in the run name, so those jobs are old six-field runs.')
    print('Pull the env-aware config generator, rerun prepare --print-table, and only then resubmit four-field training.')

run_names = [r for r in log_df.get('run_name_from_log', pd.Series(dtype=object)).dropna().tolist()]
for preferred in PREFERRED_RUNS:
    if preferred in manifest_by_run and preferred not in run_names:
        run_names.append(preferred)
run_names = list(dict.fromkeys(run_names))

rows = []
for run_name in run_names:
    row = manifest_by_run.get(run_name, {})
    ckpt_epoch, ckpt_path = latest_checkpoint_epoch(row.get('checkpoint_dir'))
    sample_path = sample_path_for_run(run_name)
    rows.append({
        'run_name': run_name,
        **audit_fields(run_name, row),
        'n_train_sims': row.get('n_train_simulations_per_field'),
        'dataset_size': row.get('dataset_size'),
        'target_updates': row.get('target_updates'),
        'actual_updates': row.get('actual_updates'),
        'epochs_target': row.get('epochs'),
        'latest_ckpt_epoch': ckpt_epoch,
        'latest_ckpt_path': ckpt_path,
        'sample_exists': sample_path.exists(),
        'sample_path': str(sample_path),
    })
run_df = pd.DataFrame(rows)
print('runs to inspect:')
display(run_df)



## Sampling commands if DPM50 files are missing

Only run these after the field audit says `OK: intended four-field run`.

The balanced sample file is valid for one-point PDFs and auto-power. For physical generated cross-correlation, also generate one class-only file per label using the same seed. Those class-only files are what make generated `Mcdm`, `HI`, `Mgas`, and `ne` matched by latent/noise index.


In [ ]:

def format_train_env(row: dict[str, Any]) -> str:
    fields = ','.join(row.get('fields', EXPECTED_FIELDS))
    n_train = row.get('n_train_simulations_per_field')
    target = row.get('target_updates')
    parts = [
        f"NF_CLASS_FIELDS='{fields}'",
        'NF_CLASS_NORMALIZATION_MODE=per-field',
    ]
    if n_train is not None:
        parts.append(f'NF_CLASS_N_TRAIN_SIMS={int(n_train)}')
    if target is not None:
        parts.append(f'NF_CLASS_TARGET_UPDATES={int(target)}')
    return ' \\\n'.join(parts)

for run_name in run_names:
    row = manifest_by_run.get(run_name)
    if not row:
        print('\n# No manifest row for', run_name)
        print('# Run prepare first after pulling the latest code.')
        continue
    audit = audit_fields(run_name, row)
    sample_path = sample_path_for_run(run_name)
    print('\n#', run_name)
    print('# audit:', audit['verdict'])
    print('# sample exists:', sample_path.exists(), sample_path)
    if not audit['matches_expected_fourfield']:
        print('# NOT printing sample commands for this run because it is not the intended four-field run.')
        continue
    env = format_train_env(row)
    print(f"""cd /home/jiamingp/diffusion_models_repo

{env} \\
SAMPLER_CLASS=DPMSolverMultistepScheduler \\
SAMPLER_STEPS=50 \\
SAMPLE_LABEL=dpm50_class_conditional \\
OVERWRITE=1 \\
sbatch -A huterer2 scripts/slurm/sample_nf_class_conditional_u128.sbatch

# Matched class-only same-seed files for physical cross-correlation:
{env} \\
SAMPLER_CLASS=DPMSolverMultistepScheduler \\
SAMPLER_STEPS=50 \\
SAMPLE_LABEL=dpm50_class_conditional \\
OVERWRITE=1 \\
sbatch -A huterer2 --array=0-{int(row.get('num_classes', len(EXPECTED_FIELDS))) - 1} scripts/slurm/sample_nf_class_conditional_u128.sbatch
""")


In [ ]:
def load_npz_array(path: Path) -> np.ndarray:
    z = np.load(path)
    for key in ('samples', 'arr_0', 'images'):
        if key in z:
            return np.asarray(z[key], dtype=np.float32)
    # fallback: first non-scalar array
    for key in z.files:
        arr = np.asarray(z[key])
        if arr.ndim >= 3:
            return arr.astype(np.float32)
    raise ValueError(f'No image array found in {path}; keys={z.files}')


def class_map_for_row(row: dict[str, Any]) -> dict[int, str]:
    cm = row.get('class_map') or {}
    if not cm and row.get('class_map_path'):
        p = PROJECT_DIR / row['class_map_path'] if not str(row['class_map_path']).startswith('/') else Path(row['class_map_path'])
        if p.exists():
            cm = json.loads(p.read_text())
    return {int(v): str(k) for k, v in cm.items()}


def labels_for_row(row: dict[str, Any]) -> np.ndarray:
    p = row.get('sample_label_path')
    if p is None:
        raise ValueError('manifest row has no sample_label_path')
    p = PROJECT_DIR / p if not str(p).startswith('/') else Path(p)
    return np.load(p)


def split_generated_by_field(samples: np.ndarray, labels: np.ndarray, id_to_field: dict[int, str]) -> dict[str, np.ndarray]:
    if samples.ndim == 3:
        samples = samples[:, None]
    return {field: samples[labels == class_id] for class_id, field in id_to_field.items()}


def apply_training_normalization(arr: np.ndarray, norm: str, kwargs: dict[str, Any]) -> np.ndarray:
    norm = str(norm).lower()
    if norm in {'none', 'false'}:
        return arr.astype(np.float32, copy=False)
    if norm == 'tanh':
        center = kwargs.get('center')
        if center is None:
            center = float(arr.mean())
        arr = arr - np.float32(center)
        xmax = kwargs.get('xmax')
        if xmax is None:
            xmax = float(np.abs(arr).max())
        arr = arr / np.float32(max(float(xmax), 1e-30))
        alpha = float(kwargs.get('alpha', 1.0))
        beta = float(kwargs.get('beta', 1.0))
        gamma = float(kwargs.get('gamma', 1.0))
        delta = float(kwargs.get('delta', 1.0))
        sigma = float(kwargs.get('sigma', 1.0))
        mu = float(kwargs.get('mu', 0.0))
        shifted = arr - np.float32(mu)
        pos = alpha * np.tanh((gamma * shifted) / alpha)
        neg = beta * np.tanh((delta * shifted) / beta)
        return (np.where(shifted >= 0, pos, neg) * sigma).astype(np.float32, copy=False)
    raise ValueError(f'unsupported normalization: {norm}')


def load_real_by_field(row: dict[str, Any], max_sims: int = REAL_SIMS_PER_FIELD) -> dict[str, np.ndarray]:
    cfg_path = PROJECT_DIR / row['config'] if not str(row['config']).startswith('/') else Path(row['config'])
    cfg = yaml.safe_load(cfg_path.read_text())
    data = cfg['data']
    fields = list(row['fields'])
    out = {}
    norms = data['normalization'] if isinstance(data['normalization'], list) else [data['normalization']] * len(fields)
    norm_kwargs = data['norm_kwargs'] if isinstance(data['norm_kwargs'], list) else [data['norm_kwargs']] * len(fields)
    transforms = data.get('transform', [[]] * len(fields))
    if not isinstance(transforms[0], list):
        transforms = [transforms] * len(fields)
    zthin = int(data.get('zthin', 1))
    for i, field in enumerate(fields):
        p = Path(data['img_path'][i])
        arr = np.array(np.load(p, mmap_mode='r')[:max_sims], dtype=np.float32, copy=True)
        if 'log' in transforms[i]:
            arr = np.log(np.maximum(arr, np.float32(1e-30)))
        arr = apply_training_normalization(arr, norms[i], dict(norm_kwargs[i] or {}))
        arr = arr[:, ::zthin].reshape(-1, 1, arr.shape[-2], arr.shape[-1])
        out[field] = np.asarray(arr, dtype=np.float32)
    return out


def step_density(ax, arr: np.ndarray, bins: np.ndarray, label: str, color: str) -> None:
    values = np.asarray(arr, dtype=np.float32).ravel()
    hist, edges = np.histogram(values[np.isfinite(values)], bins=bins, density=True)
    centers = 0.5 * (edges[:-1] + edges[1:])
    ax.step(centers, hist, where='mid', lw=2.2, color=color, label=label)


def robust_limits(*arrays: np.ndarray) -> tuple[float, float]:
    vals = np.concatenate([np.asarray(a).ravel() for a in arrays if a is not None and len(a)])
    vals = vals[np.isfinite(vals)]
    if len(vals) == 0:
        return -1.0, 1.0
    lo, hi = np.quantile(vals, [0.001, 0.999])
    pad = 0.04 * max(float(hi - lo), 1e-6)
    return float(lo - pad), float(hi + pad)

In [ ]:
def plot_onepoint_for_run(run_name: str) -> None:
    row = manifest_by_run.get(run_name)
    if not row:
        print('missing manifest row:', run_name)
        return
    sample_path = sample_path_for_run(run_name)
    if not sample_path.exists():
        print('missing sample:', sample_path)
        return
    samples = load_npz_array(sample_path)
    labels = labels_for_row(row)
    id_to_field = class_map_for_row(row)
    gen_by_field = split_generated_by_field(samples, labels, id_to_field)
    real_by_field = load_real_by_field(row)
    fields = [id_to_field[i] for i in sorted(id_to_field)]

    fig, axes = plt.subplots(1, len(fields), figsize=(4.5 * len(fields), 4.1), constrained_layout=True)
    if len(fields) == 1:
        axes = [axes]
    summary = []
    for ax, field in zip(axes, fields):
        real = real_by_field[field]
        gen = gen_by_field[field]
        lo, hi = robust_limits(real, gen)
        bins = np.linspace(lo, hi, 160)
        step_density(ax, real, bins, f'real n={len(real)}', '#222222')
        step_density(ax, gen, bins, f'generated n={len(gen)}', '#d95f02')
        ax.set_yscale('log')
        ax.set_title(field)
        ax.set_xlabel('Normalized value')
        ax.set_ylabel('Density')
        ax.legend(frameon=False)
        summary.append({
            'run_name': run_name,
            'field': field,
            'n_real': len(real),
            'n_gen': len(gen),
            'real_mean': float(real.mean()),
            'gen_mean': float(gen.mean()),
            'real_std': float(real.std()),
            'gen_std': float(gen.std()),
        })
    fig.suptitle(f'{run_name}\nGenerated vs real one-point PDFs', y=1.08)
    out = OUT_DIR / f'{run_name}_one_point.png'
    fig.savefig(out, dpi=220, bbox_inches='tight')
    print('wrote', out)
    plt.show()
    display(pd.DataFrame(summary).round(5))


def plot_pk_for_run(run_name: str) -> None:
    row = manifest_by_run.get(run_name)
    if not row:
        print('missing manifest row:', run_name)
        return
    sample_path = sample_path_for_run(run_name)
    if not sample_path.exists():
        print('missing sample:', sample_path)
        return
    samples = load_npz_array(sample_path)
    labels = labels_for_row(row)
    id_to_field = class_map_for_row(row)
    gen_by_field = split_generated_by_field(samples, labels, id_to_field)
    real_by_field = load_real_by_field(row)
    fields = [id_to_field[i] for i in sorted(id_to_field)]

    fig, axes = plt.subplots(1, len(fields), figsize=(4.5 * len(fields), 4.1), constrained_layout=True)
    if len(fields) == 1:
        axes = [axes]
    rows = []
    for ax, field in zip(axes, fields):
        pk_real, kbins = batch_power_spectra(real_by_field[field], nbins=PK_NBINS)
        pk_gen, _ = batch_power_spectra(gen_by_field[field], nbins=PK_NBINS)
        real_mean = np.nanmean(pk_real, axis=0)
        gen_mean = np.nanmean(pk_gen, axis=0)
        ratio = gen_mean / np.maximum(real_mean, 1e-30)
        ax.axhline(1.0, color='#777777', lw=1.2, ls=':')
        ax.plot(kbins, ratio, color='#1b5eae', lw=2.4)
        ax.set_xscale('log')
        ax.set_yscale('log')
        ax.set_title(field)
        ax.set_xlabel('k')
        ax.set_ylabel('Generated / real P(k)')
        rows.append({
            'run_name': run_name,
            'field': field,
            'pk_ratio_log10_mae': float(np.nanmean(np.abs(np.log10(np.maximum(ratio, 1e-30))))),
        })
    fig.suptitle(f'{run_name}\nGenerated vs real auto-power ratios', y=1.08)
    out = OUT_DIR / f'{run_name}_pk_ratio.png'
    fig.savefig(out, dpi=220, bbox_inches='tight')
    print('wrote', out)
    plt.show()
    display(pd.DataFrame(rows).round(5))


for run_name in run_names:
    print('\n===', run_name, '===')
    plot_onepoint_for_run(run_name)
    plot_pk_for_run(run_name)


## Matched Cross-Correlation Diagnostics

Auto-power above checks each field independently. Cross-correlation is different: it asks whether generated fields have the same paired multi-field structure as CAMELS.

For real data, this notebook pairs fields by the same CAMELS simulation/slice order. For generated data, the balanced class sample file is **not** a physical multi-field realization. The notebook therefore uses class-only same-seed files if they exist. If they do not exist, it skips the generated cross-correlation plot by default.

Set `PLOT_ORDER_FALLBACK_CROSS=1` only if you want a deliberately non-physical fallback for debugging labels and shapes.


In [ ]:

def radial_cross_power_spectrum_2d(a: np.ndarray, b: np.ndarray, nbins: int = PK_NBINS) -> tuple[np.ndarray, np.ndarray]:
    fa = np.asarray(a, dtype=np.float32)
    fb = np.asarray(b, dtype=np.float32)
    if fa.shape != fb.shape or fa.ndim != 2:
        raise ValueError(f'Expected matched 2D fields, got {fa.shape} and {fb.shape}.')
    fa = fa - np.nanmean(fa)
    fb = fb - np.nanmean(fb)
    fft_a = np.fft.rfft2(fa)
    fft_b = np.fft.rfft2(fb)
    cross = (fft_a * fft_b.conj()).real / fa.size
    ky = np.fft.fftfreq(fa.shape[0])
    kx = np.fft.rfftfreq(fa.shape[1])
    kk = np.sqrt(ky[:, None] ** 2 + kx[None, :] ** 2)
    valid = kk > 0
    edges = np.geomspace(max(float(kk[valid].min()), 1e-6), float(kk[valid].max()), nbins + 1)
    centers = np.sqrt(edges[:-1] * edges[1:])
    pk = np.full(nbins, np.nan, dtype=np.float64)
    for i in range(nbins):
        mask = (kk >= edges[i]) & (kk < edges[i + 1])
        if mask.any():
            pk[i] = np.nanmean(cross[mask])
    return pk, centers


def batch_cross_power_spectra(a: np.ndarray, b: np.ndarray, nbins: int = PK_NBINS) -> tuple[np.ndarray, np.ndarray]:
    aa = np.asarray(a, dtype=np.float32)
    bb = np.asarray(b, dtype=np.float32)
    if aa.ndim == 3:
        aa = aa[:, None]
    if bb.ndim == 3:
        bb = bb[:, None]
    n = min(len(aa), len(bb))
    rows = []
    kbins = None
    for ia, ib in zip(aa[:n], bb[:n]):
        pk, kbins = radial_cross_power_spectrum_2d(ia[0], ib[0], nbins=nbins)
        rows.append(pk)
    return np.asarray(rows, dtype=np.float64), np.asarray(kbins, dtype=np.float64)


def paired_truncate(by_field: dict[str, np.ndarray], fields: list[str]) -> dict[str, np.ndarray]:
    n = min(len(by_field[field]) for field in fields if field in by_field)
    return {field: np.asarray(by_field[field])[:n] for field in fields if field in by_field}


def load_matched_generated_class_only(row: dict[str, Any], fields: list[str]) -> tuple[dict[str, np.ndarray], dict[str, Path]]:
    run_name = row['run_name']
    id_to_field = class_map_for_row(row)
    by_field = {}
    paths = {}
    for class_id in sorted(id_to_field):
        field = id_to_field[class_id]
        if field not in fields:
            continue
        candidates = [
            SAMPLE_DIR / f'{run_name}_seed{SEED}_dpm50_class{class_id}_same_seed.npz',
            SAMPLE_DIR / f'{run_name}_seed{SEED}_raw_class{class_id}_same_seed.npz',
        ]
        candidates.extend(sorted(SAMPLE_DIR.glob(f'{run_name}_seed{SEED}_*class{class_id}*same_seed*.npz')))
        candidates.extend(sorted(SAMPLE_DIR.glob(f'{run_name}_seed{SEED}_*class{class_id}*.npz')))
        chosen = next((c for c in candidates if c.exists()), None)
        if chosen is None:
            return {}, {}
        by_field[field] = load_npz_array(chosen)
        paths[field] = chosen
    if set(by_field) != set(fields):
        return {}, {}
    return paired_truncate(by_field, fields), paths


def mean_rxy_curve(by_field: dict[str, np.ndarray], field_x: str, field_y: str, nbins: int = PK_NBINS) -> tuple[np.ndarray, np.ndarray]:
    pxy, kbins = batch_cross_power_spectra(by_field[field_x], by_field[field_y], nbins=nbins)
    pxx, _ = batch_cross_power_spectra(by_field[field_x], by_field[field_x], nbins=nbins)
    pyy, _ = batch_cross_power_spectra(by_field[field_y], by_field[field_y], nbins=nbins)
    mean_pxy = np.nanmean(pxy, axis=0)
    mean_pxx = np.nanmean(pxx, axis=0)
    mean_pyy = np.nanmean(pyy, axis=0)
    denom = np.sqrt(np.maximum(mean_pxx * mean_pyy, 1e-30))
    return kbins, mean_pxy / denom


def generated_cross_fields_for_run(row: dict[str, Any], fields: list[str]) -> tuple[dict[str, np.ndarray], str, dict[str, Path]]:
    matched, paths = load_matched_generated_class_only(row, fields)
    if matched:
        return matched, 'same-seed class-only generated fields', paths
    if not PLOT_ORDER_FALLBACK_CROSS:
        return {}, 'missing same-seed class-only files', {}
    sample_path = sample_path_for_run(row['run_name'])
    if not sample_path.exists():
        return {}, 'missing balanced generated file', {}
    samples = load_npz_array(sample_path)
    labels = labels_for_row(row)
    id_to_field = class_map_for_row(row)
    return paired_truncate(split_generated_by_field(samples, labels, id_to_field), fields), 'ORDER-PAIRED FALLBACK, NOT PHYSICAL', {}


def plot_rxy_for_run(run_name: str) -> None:
    row = manifest_by_run.get(run_name)
    if not row:
        print('missing manifest row:', run_name)
        return
    fields = list(row.get('fields') or fields_from_run_name(run_name))
    if not fields:
        print('cannot infer fields for', run_name)
        return
    audit = audit_fields(run_name, row)
    if not audit['matches_expected_fourfield']:
        print(f"{run_name}: {audit['verdict']}")
        print('Skipping four-field cross-correlation plot for this run.')
        return

    real_by_field = paired_truncate(load_real_by_field(row), fields)
    gen_by_field, source, paths = generated_cross_fields_for_run(row, fields)
    if not gen_by_field:
        print(f'{run_name}: no physical generated cross-correlation sample found ({source}).')
        print('Generate matched class-only files with the --array command printed above, then rerun this cell.')
        return

    print('generated cross-correlation source:', source)
    if paths:
        for field, path in paths.items():
            print(f'  {field}: {path}')

    pairs = [(fields[i], fields[j]) for i in range(len(fields)) for j in range(i + 1, len(fields))]
    ncols = 3
    nrows = int(math.ceil(len(pairs) / ncols))
    fig, axes = plt.subplots(nrows, ncols, figsize=(5.0 * ncols, 3.8 * nrows), squeeze=False, constrained_layout=True)
    rows = []
    mae_matrix = np.zeros((len(fields), len(fields)), dtype=float)
    for ax, (field_x, field_y) in zip(axes.ravel(), pairs):
        kbins, real_r = mean_rxy_curve(real_by_field, field_x, field_y)
        _, gen_r = mean_rxy_curve(gen_by_field, field_x, field_y)
        err = float(np.nanmean(np.abs(gen_r - real_r)))
        i, j = fields.index(field_x), fields.index(field_y)
        mae_matrix[i, j] = mae_matrix[j, i] = err
        ax.axhline(0, color='#777777', lw=1.0, ls=':')
        ax.plot(kbins, real_r, color='#222222', lw=2.4, label='real paired')
        ax.plot(kbins, gen_r, color='#d95f02', lw=2.4, label='generated matched')
        ax.set_xscale('log')
        ax.set_ylim(-1.05, 1.05)
        ax.set_title(f'{field_x} x {field_y}')
        ax.set_xlabel('k')
        ax.set_ylabel(r'$r_{XY}(k)$')
        ax.legend(frameon=False)
        rows.append({
            'run_name': run_name,
            'field_x': field_x,
            'field_y': field_y,
            'rxy_mae': err,
            'real_rxy_mean': float(np.nanmean(real_r)),
            'gen_rxy_mean': float(np.nanmean(gen_r)),
            'generated_cross_power_source': source,
        })
    for ax in axes.ravel()[len(pairs):]:
        ax.axis('off')
    fig.suptitle(f'{run_name}\nMatched generated vs real paired cross-correlation curves', y=1.06)
    out = OUT_DIR / f'{run_name}_matched_rxy_curves.png'
    fig.savefig(out, dpi=220, bbox_inches='tight')
    print('wrote', out)
    plt.show()

    fig, ax = plt.subplots(figsize=(6.0, 5.1), constrained_layout=True)
    im = ax.imshow(mae_matrix, vmin=0, vmax=max(1.0, float(np.nanmax(mae_matrix))), cmap='magma_r')
    ax.set_xticks(range(len(fields)), fields, rotation=35, ha='right')
    ax.set_yticks(range(len(fields)), fields)
    ax.set_xlabel('field Y')
    ax.set_ylabel('field X')
    ax.set_title(r'$r_{XY}(k)$ MAE, generated vs real (0 is better)')
    fig.colorbar(im, ax=ax, label='MAE')
    out = OUT_DIR / f'{run_name}_matched_rxy_mae_heatmap.png'
    fig.savefig(out, dpi=220, bbox_inches='tight')
    print('wrote', out)
    plt.show()

    summary = pd.DataFrame(rows)
    display(summary.round(5))
    csv = OUT_DIR / f'{run_name}_matched_rxy_summary.csv'
    summary.to_csv(csv, index=False)
    print('wrote', csv)


for run_name in run_names:
    print('\n=== cross-correlation:', run_name, '===')
    plot_rxy_for_run(run_name)



## Interpretation notes

- If `run_name_from_log` contains `mstar` or `mtot`, then the cluster checkout did **not** use the new four-field env-aware config generator. Do not interpret that job as the new four-field run.
- The balanced `dpm50_class_conditional` sample file is valid for one-point PDFs and auto-power `P_XX(k)`.
- For generated cross-correlation `r_XY(k)`, use class-only same-seed files from the optional array sampling command. Otherwise generated cross-field pairing is not a physical multi-field realization.
- The cross-correlation plot in this notebook is deliberately the PDF-style scale-dependent curve `r_XY(k) = P_XY(k) / sqrt(P_XX(k) P_YY(k))`; it is not the older sign/magnitude split heatmap.
